### 1) API call to get statewide GDOT projects


In [ ]:
# depencencies
import requests
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import numpy as np
import glob
import os

# Define the base URL of the MapServer layer
base_url = "https://rnhp.dot.ga.gov/hosting/rest/services/GDOT_Public_Outreach/Hub_Project_Search/MapServer/2/query"

# Define query parameters
params = {
    # filter by construction status to include both under construction and pre-construction
    "where": "CONSTRUTION_STATUS_DERIVED LIKE '%'",
    "outFields": "*",  # Get all fields
    "f": "geojson",  # Request data in GeoJSON format
    "returnGeometry": "true",  # Include geometry data
    "resultOffset": 0,  # Start from the first record
    "resultRecordCount": 1000  # Get 1000 records
}

# list to store each batch of GeoDataFrames
gdfs = []

while True:

    # Make the GET request
    response = requests.get(base_url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        batch_gdf = gpd.GeoDataFrame.from_features(data["features"])

        # if no more data is returned, break the loop
        if batch_gdf.empty:
            break

        gdfs.append(batch_gdf)

        print(
            f"Fetched {len(batch_gdf)} records (offset: {params['resultOffset']})")

        # increase offset by the batch size to fetch the next chunk
        params["resultOffset"] += params["resultRecordCount"]

    else:
        print(
            f"❌ Error fetching data: {response.status_code} - {response.text}")
        break

# Filter out empty GeoDataFrames
gdfs = [gdf.dropna(axis=1, how='all') for gdf in gdfs if not gdf.empty]

# Combine all batches into a single GeoDataFrame
if gdfs:
    gdot_projects = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

else:
    print("❌ No data retrieved.")

# drop any columns that ONLY contain NaN values
gdot_projects = gdot_projects.dropna(axis=1, how="all")

# replace spaces in the 'CONSTRUTION_STATUS_DERIVED' column with hyphens
gdot_projects['CONSTRUTION_STATUS_DERIVED'] = gdot_projects['CONSTRUTION_STATUS_DERIVED'].str.replace(
    " ", "-")
gdot_projects['CONSTRUCTION_STATUS_DERIVED'] = gdot_projects['CONSTRUTION_STATUS_DERIVED'].str.strip()

# if the 'CONTRACT_DESCRIPTION' column is empty, replace it with the 'SHORT_DESCR' column
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].fillna(
    gdot_projects['SHORT_DESCR'])

# define a list of sub-strings to keep in all caps; everything else in the 'SHORT_DESCR' column should be converted to title case
keep_all_caps = ["SR ", "CO ", "CR ", "US ", "RD", "SO ",
                 "CS ", "MI ", "SE ", "NE ", "SW ", "NW ",
                 "NS ", "CSX", "CCTV", "-TIA", " - TIA", "LCI", "GRTA"]

# convert to title case
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.title()
for sub_str in keep_all_caps:
    gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
        sub_str.title(), sub_str)

# clean up some of the description sub-strings
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Brdg", "Bridge")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Mtn ", "Mountain ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Ii", "Phase II")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Ph Ii", "Phase II")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase IIi", "Phase III")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Ph IIi", "Phase III")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Iv", "Phase IV")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Vi", "Phase VI")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Resf ", "Resurface ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Rsrf ", "Resurface ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Shldr ", "Shoulder ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Recnst", "Reconstruction")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Cnst ", "Construction ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Fm ", "From ")

# filter out records that have either 'WPH' (completed) or 'REJ' (rejected) in the 'CONST_STAT_CD' column
gdot_projects = gdot_projects[~gdot_projects['CONST_STAT_CD'].isin([
                                                                   'WPH', 'REJ'])]

# rename columns
gdot_projects = gdot_projects.rename(columns={
    'LAST_REFRESH_DTTM': 'Last_refresh',
    'PROJECT_COUNTIES': 'Project_counties',
    'PROJ_ID': 'Project_ID',
    'CONTRACT_DESCRIPTION': 'Project_description',
    'CONTRACTOR_NAME': 'Contractor_name',
    'IS_TIA_PROJECT': 'Is_TIA_project',
    'CONSTRUTION_STATUS_DERIVED': 'CONSTRUCTION_STATUS_DERIVED'
})

# drop unneeded columns
gdot_projects = gdot_projects.drop(columns=[
    'OBJECTID',
    'PRIORITY_CD',
    'SOURCE_OF_CONSTRUCTION_DATES',
    'CONTRACT_ID',
    'CONSTRUTION_STATUS_DERIVED_RSN',
    'PAYMENT_PERCENT_COMPLETE',
    'ESRI_OID',
    'SHORT_DESCR',
    'REC_STATUS',
    'LET_RESP_CD',
    'PRIORITY_CD_DESCR',
    'CONST_STAT_CD',
    'CONSTRUCTION_PERCENT_COMPLETE',
    'CURR_COMPLETION_DATE',
    'AWARD_DATE'
])

# create URL column
gdot_projects['Project_URL'] = 'https://www.dot.ga.gov/applications/geopi/Pages/Dashboard.aspx?ProjectId=' + \
    gdot_projects['Project_ID'].astype(str)

# Read in 2023 congressional districts
gdot_projects = gdot_projects.set_crs(epsg=4326)
districts = gpd.read_file(
    'data/congressional_districts/cdistricts.geojson')
districts = districts.to_crs(epsg=4326)
districts = districts[[
    'DISTRICT',
    'geometry'
]]

# spatial join projects to get congressional district
gdot_projects = gpd.sjoin(
    gdot_projects,
    districts,
    how='left',
    predicate='intersects'
).drop(columns=['index_right'])

# convert 'DISTRICT' column to integer
gdot_projects['DISTRICT'] = gdot_projects['DISTRICT'].astype(int)

# drop any duplicate columns
gdot_projects = gdot_projects.T.drop_duplicates().T

# export the GeoDataFrame to a CSV file
csv_export = gdot_projects.drop(columns=['geometry']).to_csv(
    'GDOT_export.csv', index=False)

# export timestamp file to be inserted in <div> on frontend
current_date = datetime.now().strftime("%B %d, %Y")
with open("data/current_date.txt", "w") as f:
    f.write(current_date)

print("✅ Project data & timestamp exported.")

Fetched 1000 records (offset: 0)
Fetched 1000 records (offset: 1000)
Fetched 1000 records (offset: 2000)
Fetched 1000 records (offset: 3000)
Fetched 1000 records (offset: 4000)
Fetched 1000 records (offset: 5000)
Fetched 12 records (offset: 6000)
✅ Successfully exported 4,002 records!


,geometry,Project_ID,Project_counties,Is_TIA_project,Contractor_name,TPRO_PROJ_COMPLETE_DT,Project_description,TIME_STOPPED_DATE,SUBSTL_WORK_COMPL_DATE,CONSTRUCTION_STATUS_DERIVED,Last_refresh,Project_URL,DISTRICT
0,LINESTRING (-84.225693991106 33.54372026819995...,0015090,Henry,No,None,None,Rock Quarry Road From Eagles Landing Pkwy To S...,None,None,PRE-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,13
1,LINESTRING (-83.92104177199927 33.615142716665...,0015095,Newton,No,SUMMIT CONSTRUCTION & DEV,None,Ca - Access RD @ I-20 From W Of Crowell RD To,None,None,UNDER-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,13
2,LINESTRING (-83.90086899736657 33.586828640390...,0015096,Newton,No,WRIGHT BROTHERS CST CO,None,Ca - CR 511/Brown Bridge RD At Yellow Creek,None,None,UNDER-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,13
3,LINESTRING (-83.98986581660641 33.568463044134...,0015097,Newton,No,CMES INC,None,CR 511/Brown Bridge RD At Snapping Shoals Crk;,None,None,UNDER-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,13
5,LINESTRING (-83.98865160875263 33.587754014885...,0015100,Rockdale,No,WILLIAMS CONTRACTING CO,None,Honey Creek RD (CR 505) - Construction Of A Br...,None,None,UNDER-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6007,LINESTRING (-83.82880495231181 34.164408938612...,M006610,Hall,No,None,None,SR 53 From SR 211 To Jackson County Line,None,None,PRE-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,9
6008,LINESTRING (-82.36629293541498 32.586632175461...,M006611,Emanuel,No,None,None,SR 26 From W Of SR 4 To W Of SR 56,None,None,PRE-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,12
6009,LINESTRING (-81.97489427836476 33.480134349927...,M006612,Richmond,No,None,None,SR 104 & SR 104Ea From SR 4 To 15Th Street,None,None,PRE-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,12
6010,LINESTRING (-82.40910855347103 32.353892580366...,M006613,"Emanuel , Treutlen",No,None,None,SR 297 From Toombs County Line To Ohoopee River,None,None,PRE-CONSTRUCTION,1742810130000,https://www.dot.ga.gov/applications/geopi/Page...,12


In [ ]:
INPUT_CSV = "GDOT_export.csv"
df = pd.read_csv(INPUT_CSV)

# keep only the first instance of each unique 'Parcl_ID'
df = df.drop_duplicates(subset=['Parcl_ID'], keep='first')

3833

### 2) Scrape exported GDOT projects to get additional information


#### old scraping code


In [3]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

# Input and output file paths
INPUT_CSV = "GDOT_export.csv"
OUTPUT_CSV = "gdot_scraped_data.csv"
districts = [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
]

df = pd.read_csv(INPUT_CSV)

# loop through each district and scrape project data
for district in districts:
    df = df[df['DISTRICT'] == district]

    # Get the total number of projects
    total_projects = len(df)

    # Initialize the project index
    project_index = 0

    # Create an empty list to store scraped data
    all_data = []

    async def scrape_project_data():
        global project_index

        # Start async Playwright
        async with async_playwright() as p:
            # Launch a browser (headless mode for speed)
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()

            # Loop through each project URL
            for index, row in df.iterrows():
                project_id = row["Project_ID"]
                project_url = row["Project_URL"]
                project_index += 1

                # Calculate the percentage
                percentage = (project_index / total_projects) * 100
                print(
                    f"Scraping Project {project_id} in District {district} ({project_index} of {total_projects}, or {percentage:.1f}% complete)")

                # Open the project page
                await page.goto(project_url)

                # Select all rows inside the tbody
                rows = await page.query_selector_all("table.rgMasterTable tbody tr")

                # Wait for the table to load before scraping
                try:
                    await page.wait_for_selector("table.rgMasterTable", timeout=15000)
                except:
                    print(
                        f"Table not found for Project ID {project_id}. Skipping...")
                    continue

                # ================================
                # ✅ Scrape Project Description
                # ================================
                try:
                    # Get all rows in the ProjectDescriptionTable
                    description_rows = await page.query_selector_all("table.ProjectDescriptionTable tbody tr")

                    # Check if there is a second <tr> (index 1), then get its <td>
                    if len(description_rows) >= 2:
                        second_row_cells = await description_rows[1].query_selector_all("td")
                        if second_row_cells:
                            project_description = (await second_row_cells[0].inner_text()).strip()
                        else:
                            project_description = "N/A"
                    else:
                        project_description = "N/A"

                except Exception:
                    project_description = "N/A"

                # ==========================================
                # ✅ Scrape Project Manager & Completion Date
                # ==========================================
                try:
                    project_info_rows = await page.query_selector_all("table.ProjectInformationTable tbody tr")

                    # Check if the 3rd row (index 2) exists and has 4 <td> elements
                    if len(project_info_rows) >= 3:
                        third_row_cells = await project_info_rows[2].query_selector_all("td")
                        if len(third_row_cells) >= 4:
                            project_manager = (await third_row_cells[1].inner_text()).strip()
                            completion_date = (await third_row_cells[3].inner_text()).strip()
                        else:
                            project_manager, completion_date = "N/A", "N/A"
                    else:
                        project_manager, completion_date = "N/A", "N/A"
                except Exception:
                    project_manager, completion_date = "N/A", "N/A"

                # ================================
                # ✅ Scrape Lower Table
                # ================================

                # Get all rows from the table's <tbody> section
                rows = await page.query_selector_all("table.rgMasterTable tbody tr")

                for row in rows:
                    cells = await row.query_selector_all("td")
                    cell_data = [await cell.inner_text() for cell in cells]

                    # Clean non-breaking spaces and weird characters
                    clean_data = [cell.replace("\xa0", " ").replace(
                        "¬†", "").strip() for cell in cell_data]

                    # Add Project_ID and Project_URL to the start of the row
                    if len(clean_data) == 4:
                        all_data.append([
                            project_id, project_url, *clean_data,
                            project_description, project_manager, completion_date
                        ])

            # Close the browser after scraping
            await browser.close()

    # Run the scraper
    asyncio.run(scrape_project_data())

    # Define the column headers
    columns = [
        "Project_ID", "Project_URL", "Activity",
        "Program Year", "Cost Estimate", "Date of Last Estimate",
        "Project_description", "Project_manager", "Completion_date"
    ]

    # Create DataFrame and drop duplicate/empty rows
    clean_df = pd.DataFrame(all_data, columns=columns)

    # Drop empty rows or unnecessary ones
    clean_df = clean_df.replace("", pd.NA).dropna(
        how="all", subset=columns[2:])

    # reformat the 'Cost Estimate' column
    clean_df['Cost Estimate'] = clean_df['Cost Estimate'].fillna(0)
    clean_df['Cost Estimate'] = clean_df['Cost Estimate'].str.replace(
        '$', '').str.replace(',', '').astype(float)

    # Group by Project_ID, summing "Cost Estimate" while keeping the first occurrence of other columns
    final_df = clean_df.groupby("Project_ID", as_index=False).agg(
        Project_URL=("Project_URL", "first"),
        Total_Cost=("Cost Estimate", "sum"),
        Project_manager=("Project_manager", "first"),
        Description=("Project_description", "first"),
    )

    # Save cleaned data to a new CSV
    final_df.to_csv(
        f"scraped/GDOT_scraped_District{district}.csv", index=False)

    print(f"✅ Scraping complete for District {district}!")

RuntimeError: asyncio.run() cannot be called from a running event loop

#### new scraping code


In [ ]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

# Input and output file paths
INPUT_CSV = "GDOT_export.csv"
OUTPUT_DIR = "scraped/"

districts = [
    # 1,
    # 2,
    # 3,
    # 4,
    # 5,
    # 6,
    # 7,
    8,
    9,
    10,
    11,
    12,
    13,
    14
]

# Read the input CSV once
df = pd.read_csv(INPUT_CSV)
df = df.drop_duplicates(subset=['Project_ID'], keep='first')

# ================================
# ✅ Scraping Function (Outside Loop)
# ================================


async def scrape_project_data(district, df):
    """Scrape project data for a given district and return a cleaned DataFrame."""
    df_district = df[df["DISTRICT"] == district]
    total_projects = len(df_district)

    # Skip if no projects in this district
    if total_projects == 0:
        print(f"⚠️ No projects found for District {district}. Skipping...")
        return None

    all_data = []
    project_index = 0

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for index, row in df_district.iterrows():
            project_id = row["Project_ID"]
            project_url = row["Project_URL"]
            project_index += 1

            # Progress indicator
            percentage = (project_index / total_projects) * 100
            print(
                f"Scraping Project {project_id} in District {district} "
                f"({project_index} of {total_projects}, {percentage:.1f}% complete)"
            )

            # Open the project page
            await page.goto(project_url)

            # Wait for the table to load before scraping
            try:
                await page.wait_for_selector("table.rgMasterTable", timeout=15000)
            except Exception:
                print(
                    f"❌ Table not found for Project ID {project_id}. Skipping...")
                continue

            # Scrape Project Description
            project_description = await scrape_project_description(page)

            # Scrape Project Manager & Completion Date
            project_manager, completion_date = await scrape_project_info(page)

            # Scrape Lower Table Data
            lower_table_data = await scrape_lower_table(page)

            for row_data in lower_table_data:
                all_data.append(
                    [
                        project_id,
                        project_url,
                        *row_data,
                        project_description,
                        project_manager,
                        completion_date,
                    ]
                )

             # ✅ Add delay to avoid hitting the server too fast
            await asyncio.sleep(0.5)  # Wait 1 second between project scrapes

        await browser.close()

    # Return scraped data as DataFrame
    return clean_and_save_data(all_data, district)


# ================================
# ✅ Helper Functions
# ================================
async def scrape_project_description(page):
    """Scrape the project description from the description table."""
    try:
        description_rows = await page.query_selector_all("table.ProjectDescriptionTable tbody tr")
        if len(description_rows) >= 2:
            second_row_cells = await description_rows[1].query_selector_all("td")
            if second_row_cells:
                return (await second_row_cells[0].inner_text()).strip()
    except Exception:
        pass
    return "N/A"


async def scrape_project_info(page):
    """Scrape project manager and completion date from project info table."""
    try:
        project_info_rows = await page.query_selector_all("table.ProjectInformationTable tbody tr")
        if len(project_info_rows) >= 3:
            third_row_cells = await project_info_rows[2].query_selector_all("td")
            if len(third_row_cells) >= 4:
                project_manager = (await third_row_cells[1].inner_text()).strip()
                completion_date = (await third_row_cells[3].inner_text()).strip()
                return project_manager, completion_date
    except Exception:
        pass
    return "N/A", "N/A"


async def scrape_lower_table(page):
    """Scrape the lower table data from the page."""
    all_rows_data = []
    rows = await page.query_selector_all("table.rgMasterTable tbody tr")
    for row in rows:
        cells = await row.query_selector_all("td")
        cell_data = [await cell.inner_text() for cell in cells]
        clean_data = [cell.replace("\xa0", " ").replace(
            "¬†", "").strip() for cell in cell_data]
        if len(clean_data) == 4:
            all_rows_data.append(clean_data)
    return all_rows_data


def clean_and_save_data(all_data, district):
    """Clean and save scraped data to a CSV."""
    if not all_data:
        print(f"⚠️ No valid data for District {district}. Skipping save...")
        return None

    columns = [
        "Project_ID",
        "Project_URL",
        "Activity",
        "Program Year",
        "Cost Estimate",
        "Date of Last Estimate",
        "Project_description",
        "Project_manager",
        "Completion_date",
    ]

    clean_df = pd.DataFrame(all_data, columns=columns)
    clean_df = clean_df.replace("", pd.NA).dropna(
        how="all", subset=columns[2:])
    clean_df["Cost Estimate"] = (
        clean_df["Cost Estimate"].fillna(0).str.replace(
            "$", "").str.replace(",", "").astype(float)
    )

    final_df = clean_df.groupby("Project_ID", as_index=False).agg(
        Project_URL=("Project_URL", "first"),
        Total_Cost=("Cost Estimate", "sum"),
        Project_manager=("Project_manager", "first"),
        Description=("Project_description", "first"),
    )

    output_path = f"{OUTPUT_DIR}/GDOT_scraped_District{district}.csv"
    final_df.to_csv(output_path, index=False)
    print(
        f"✅ Scraping complete for District {district}. Data saved to: {output_path}")
    return final_df


# ================================
# 🚀 Main Function
# ================================
async def main():
    """Run scraping for all districts in parallel."""
    for district in districts:
        await scrape_project_data(district, df)

        # ✅ Add delay between districts
        print(f"⏳ Waiting before scraping the next district...")
        await asyncio.sleep(2)  # 5 seconds delay between districts

    print("✅ All districts scraped successfully!")


# Run the scraper
if __name__ == "__main__":
    await main()

Scraping Project 0015130 in District 2 (1 of 320, 0.3% complete)
Scraping Project 0000473 in District 2 (2 of 320, 0.6% complete)
Scraping Project 0000475 in District 2 (3 of 320, 0.9% complete)
Scraping Project 0001339 in District 2 (4 of 320, 1.2% complete)
Scraping Project 0002864 in District 2 (5 of 320, 1.6% complete)
Scraping Project 0002869 in District 2 (6 of 320, 1.9% complete)
Scraping Project 0002981 in District 2 (7 of 320, 2.2% complete)
Scraping Project 0004752 in District 2 (8 of 320, 2.5% complete)
Scraping Project 0004753 in District 2 (9 of 320, 2.8% complete)
Scraping Project 0002863 in District 2 (10 of 320, 3.1% complete)
Scraping Project 0004791 in District 2 (11 of 320, 3.4% complete)
Scraping Project 0004792 in District 2 (12 of 320, 3.8% complete)
Scraping Project 0005941 in District 2 (13 of 320, 4.1% complete)
Scraping Project 0004790 in District 2 (14 of 320, 4.4% complete)
Scraping Project 0006446 in District 2 (15 of 320, 4.7% complete)
Scraping Project 00

### 3) Prepare & export data for GeoJSON export


In [ ]:
# read in scraped data using glob
gdot_scraped = pd.concat([pd.read_csv(f) for f in glob.glob(
    "scraped/*.csv")]).drop(columns=['Unnamed: 0'], axis=1)

In [ ]:


# Ensure all properties are JSON serializable
gdot_projects = gdot_projects.replace(
    {np.nan: None, np.inf: None, -np.inf: None})

# create a new GeoJSON feature collection with a unique integer ID for each feature
features = []
for i, row in gdot_projects.iterrows():

    feature = {
        'type': 'Feature',
        'id': i + 1,  # assign a unique integer ID
        'properties': row.drop('geometry').to_dict(),
        'geometry': row['geometry'].__geo_interface__
    }
    features.append(feature)

# create a new GeoJSON feature collection
collection = {
    'type': 'FeatureCollection',
    'features': features
}

# export the feature collection to a GeoJSON file
with open("data/GDOT_export.geojson", 'w') as f:
    json.dump(collection, f)

# print status
print(f"✅ Successfully exported {len(gdot_projects):,} records to GeoJSON!")

# Display final GeoDataFrame
gdot_projects